# Brain Age Prediction: K-Fold Ensemble vs Best Fold
Questo notebook implementa il **K-Fold Ensemble** e offre un confronto diretto:
1. Addestra 5 reti separate (una per ogni Fold).
2. Salva i pesi di tutte e 5 le reti.
3. Sul Test Set incontaminato, confronta visivamente l'errore commesso dalla *Miglior Singola Rete* rispetto all'errore commesso dall'*Ensemble* (la media del comitato), per dimostrare empiricamente il vantaggio dell'Ensemble.

In [ ]:
!rm -rf SFCN
!git clone https://github.com/PietroSchgor/SFCN.git

import sys
sys.path.append('/kaggle/working/SFCN')

In [ ]:
import os
import json
import glob
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import StratifiedKFold, train_test_split

# Import dal repository
from dp_model.model_files.sfcn import SFCN
from dp_model import dp_utils as dpu
from train import train_model

PLOTS_DIR = '/kaggle/working/plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

## 1. Dataset Custom

In [ ]:
class BrainAgeDataset(Dataset):
    def __init__(self, data_dir, modality='FLAIR', is_train=True):
        self.data_dir = data_dir
        self.modality = modality 
        self.is_train = is_train
        self.subject_dirs = sorted(glob.glob(os.path.join(data_dir, "sub-*")))
        self.samples = []
        
        self.bin_range = [0, 70]
        self.bin_step = 1
        self.sigma = 1.0
        
        for subj_dir in self.subject_dirs:
            subj_id = os.path.basename(subj_dir)
            
            nii_path = os.path.join(subj_dir, f"{subj_id}_{self.modality}_MNI152_1mm.nii")
            if not os.path.exists(nii_path):
                nii_path = nii_path + ".gz"
                if not os.path.exists(nii_path):
                    if self.modality == 'T1w':
                        nii_path_alt = os.path.join(subj_dir, f"{subj_id}_T1_MNI152_1mm.nii.gz")
                        if os.path.exists(nii_path_alt):
                            nii_path = nii_path_alt
                        else:
                            continue
                    else:
                        continue
                    
            json_path = os.path.join(subj_dir, f"{subj_id}_participant_info.json")
            if not os.path.exists(json_path):
                continue
                
            with open(json_path, 'r') as f:
                info = json.load(f)
                
            participant_info = info.get("participant_info", {})
            age_cat_val = participant_info.get("age_scan")
            
            if age_cat_val is None:
                continue
            
            try:
                age_cat = int(age_cat_val) - 1
                true_age = 3 + age_cat * 5
                y, _ = dpu.num2vect(true_age, self.bin_range, self.bin_step, self.sigma)
            except:
                continue
            
            self.samples.append({
                "nii_path": nii_path,
                "label_vect": y,
                "true_age": true_age
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        img = nib.load(sample['nii_path'])
        data = img.get_fdata(dtype=np.float32)
        
        mean_val = np.mean(data)
        if mean_val > 0:
            data = data / mean_val
            
        in_sp = data.shape
        out_sp = (160, 192, 160)
        
        if self.is_train:
            if np.random.rand() > 0.5:
                data = np.flip(data, axis=0).copy()
                
            dx = np.random.randint(-2, 3)
            dy = np.random.randint(-2, 3)
            dz = np.random.randint(-2, 3)
        else:
            dx, dy, dz = 0, 0, 0
            
        x_c = int((in_sp[0] - out_sp[0]) / 2) + dx
        y_c = int((in_sp[1] - out_sp[1]) / 2) + dy
        z_c = int((in_sp[2] - out_sp[2]) / 2) + dz
        
        data = data[x_c:x_c+out_sp[0], y_c:y_c+out_sp[1], z_c:z_c+out_sp[2]]
        data = np.expand_dims(data, axis=0)
        
        tensor_data = torch.from_numpy(data)
        label_vect = torch.tensor(sample['label_vect'], dtype=torch.float32)
        
        return tensor_data, label_vect, sample['true_age']

## 2. Addestramento 5-Fold

In [ ]:
KAGGLE_DATA_DIR = "/kaggle/input/datasets/elenaschgor/dataset-2-t1-flair/ds004199_final/"
CURRENT_MODALITY = 'FLAIR'  # <-- Cambia in 'T1w' per le T1

full_train_dataset = BrainAgeDataset(KAGGLE_DATA_DIR, modality=CURRENT_MODALITY, is_train=True)
full_val_dataset   = BrainAgeDataset(KAGGLE_DATA_DIR, modality=CURRENT_MODALITY, is_train=False) 
dataset_size = len(full_train_dataset)

print(f"Trovati {dataset_size} campioni validi per {CURRENT_MODALITY}.\n")

if dataset_size > 0:
    all_ages = [sample['true_age'] for sample in full_train_dataset.samples]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Hold-out Test Set
    all_indices = np.arange(dataset_size)
    train_val_idx, test_idx = train_test_split(
        all_indices, 
        test_size=0.10, 
        random_state=42, 
        stratify=all_ages
    )
    
    k_folds = 5
    skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
    train_val_ages = np.array(all_ages)[train_val_idx]
    
    saved_models_paths = []
    fold_results = [] # Salviamo i MAE di ogni fold per identificare il "Miglior Fold"
    
    for fold, (fold_train_idx, fold_val_idx) in enumerate(skf.split(train_val_idx, train_val_ages)):
        print(f"\n==============================================")
        print(f"      ADDESTRAMENTO {CURRENT_MODALITY} - FOLD {fold + 1}/{k_folds}")
        print(f"==============================================")
        
        train_idx = train_val_idx[fold_train_idx]
        val_idx = train_val_idx[fold_val_idx]
        
        train_dataset = torch.utils.data.Subset(full_train_dataset, train_idx)
        val_dataset = torch.utils.data.Subset(full_val_dataset, val_idx)
        
        # Weighted Sampler
        fold_train_ages = [full_train_dataset.samples[i]['true_age'] for i in train_idx]
        unique_ages, counts = np.unique(fold_train_ages, return_counts=True)
        age_weight_dict = {age: 1.0 / count for age, count in zip(unique_ages, counts)}
        sample_weights = [age_weight_dict[age] for age in fold_train_ages]
        sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
        
        train_loader = DataLoader(train_dataset, batch_size=8, sampler=sampler, num_workers=2)
        val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)
        
        model = SFCN(output_dim=70)
        if torch.cuda.device_count() > 1:
            model = nn.DataParallel(model)
        model = model.to(device)
        
        optimizer = torch.optim.SGD(model.parameters(), lr=0.01, weight_decay=0.001)
        
        trained_model, train_losses, val_losses, val_maes = train_model(
            model=model, 
            train_loader=train_loader, 
            val_loader=val_loader, 
            optimizer=optimizer, 
            device=device, 
            epochs=500,
            step_size=200,
            gamma=0.3,
            patience=100,
            fold_idx=fold+1
        )
        
        best_fold_mae = min(val_maes)
        fold_results.append(best_fold_mae)
        
        fold_save_path = f"/kaggle/working/sfcn_{CURRENT_MODALITY}_fold_{fold+1}.pth"
        if isinstance(trained_model, nn.DataParallel):
            torch.save(trained_model.module.state_dict(), fold_save_path)
        else:
            torch.save(trained_model.state_dict(), fold_save_path)
            
        saved_models_paths.append(fold_save_path)
        print(f"\n[!] Modello Fold {fold+1} salvato in: {fold_save_path}")
        
    # Troviamo l'indice del Fold che ha performato meglio in validazione
    best_fold_idx = np.argmin(fold_results)
    print(f"\n>>> Il Miglior Fold singolo è stato il Fold {best_fold_idx + 1} con Val MAE: {fold_results[best_fold_idx]:.2f}")

## 3. Test Finale e Confronto (Ensemble vs Rete Singola Migliore)

In [ ]:
if dataset_size > 0:
    print("\n==============================================")
    print("   CONFRONTO TEST SET: ENSEMBLE vs BEST FOLD")
    print("==============================================")
    
    ensemble_models = []
    for path in saved_models_paths:
        m = SFCN(output_dim=70)
        m.load_state_dict(torch.load(path, map_location=device))
        m.to(device)
        m.eval()
        ensemble_models.append(m)
        
    test_dataset = torch.utils.data.Subset(full_val_dataset, test_idx)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
    
    ensemble_errors = []
    best_fold_errors = []
    
    all_true_ages = []
    all_ensemble_preds = []
    all_best_fold_preds = []
    
    bin_centers = np.arange(0, 70, 1)
    
    with torch.no_grad():
        for inputs, _, true_age in test_loader:
            inputs = inputs.to(device)
            true_age_val = true_age.item()
            
            probabilities_from_all_folds = []
            
            for m in ensemble_models:
                out = m(inputs)[0].view(1, -1)
                prob = torch.exp(out).cpu().numpy()
                probabilities_from_all_folds.append(prob)
                
            # PREDIZIONE 1: ENSEMBLE (Media di tutti e 5)
            avg_prob = np.mean(probabilities_from_all_folds, axis=0)
            ensemble_pred_age = (avg_prob @ bin_centers)[0]
            ensemble_error = abs(ensemble_pred_age - true_age_val)
            
            # PREDIZIONE 2: RETE SINGOLA MIGLIORE (Estraiamo solo le probabilità del Fold vincente)
            best_fold_prob = probabilities_from_all_folds[best_fold_idx]
            best_fold_pred_age = (best_fold_prob @ bin_centers)[0]
            best_fold_error = abs(best_fold_pred_age - true_age_val)
            
            # Salvataggio dati
            ensemble_errors.append(ensemble_error)
            best_fold_errors.append(best_fold_error)
            all_true_ages.append(true_age_val)
            all_ensemble_preds.append(ensemble_pred_age)
            all_best_fold_preds.append(best_fold_pred_age)
            
    ensemble_mae = np.mean(ensemble_errors)
    best_fold_mae = np.mean(best_fold_errors)
    
    print(f"\n>>> MAE RETE SINGOLA MIGLIORE (Fold {best_fold_idx+1}): {best_fold_mae:.2f} anni")
    print(f">>> MAE K-FOLD ENSEMBLE (Tutte e 5)       : {ensemble_mae:.2f} anni")
    
    # --- PLOT 1: SCATTER COMPARATIVO ---
    plt.figure(figsize=(14, 6))
    plt.subplot(1, 2, 1)
    plt.scatter(all_true_ages, all_best_fold_preds, color='crimson', edgecolor='k', s=60, alpha=0.6, label='Singolo Fold Migliore')
    plt.scatter(all_true_ages, all_ensemble_preds, color='teal', edgecolor='k', s=80, alpha=0.9, marker='*', label='K-Fold Ensemble')
    
    min_val = min(min(all_true_ages), min(all_ensemble_preds), min(all_best_fold_preds)) - 2
    max_val = max(max(all_true_ages), max(all_ensemble_preds), max(all_best_fold_preds)) + 2
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfetto', zorder=0)
    
    plt.title(f'Scatter Plot: Ensemble vs Best Fold ({CURRENT_MODALITY})')
    plt.xlabel('Età Reale (Anni)')
    plt.ylabel('Età Predetta (Anni)')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    
    # --- PLOT 2: BAR CHART AFFIANCATO (GROUPED BARS) ---
    plt.subplot(1, 2, 2)
    x = np.arange(len(ensemble_errors))  # the label locations
    width = 0.35  # the width of the bars
    
    plt.bar(x - width/2, best_fold_errors, width, label=f'Best Fold (MAE {best_fold_mae:.2f})', color='crimson', edgecolor='k', alpha=0.8)
    plt.bar(x + width/2, ensemble_errors, width, label=f'Ensemble (MAE {ensemble_mae:.2f})', color='teal', edgecolor='k', alpha=0.9)
    
    plt.title('Errore Assoluto per Paziente: Chi sbaglia meno?')
    plt.xlabel('Indice Paziente')
    plt.ylabel('Errore Assoluto (Anni)')
    plt.xticks(x, range(1, len(ensemble_errors) + 1))
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, f'05_ensemble_vs_bestfold_comparison_{CURRENT_MODALITY}.png'))
    plt.show()
